In [60]:
import pandas as pd
from pathlib import Path

FILE_NAME = "GB_GBN SPECIFIC DATASET.xlsx"

DATA_PATH = Path("..") / "data" / "raw" / FILE_NAME
print("File found:", DATA_PATH.exists())

df = pd.read_excel(DATA_PATH)
df["utc_timestamp"] = pd.to_datetime(df["utc_timestamp"], utc=True)


File found: True


In [61]:
#checking for missing values
df.isnull().sum()

utc_timestamp                                  0
GB_GBN_load_actual_entsoe_transparency         7
GB_GBN_load_forecast_entsoe_transparency     121
GB_GBN_price_day_ahead                       111
GB_GBN_solar_capacity                       6600
GB_GBN_solar_generation_actual                55
GB_GBN_solar_profile                        6649
GB_GBN_wind_capacity                        6600
GB_GBN_wind_generation_actual                 41
GB_GBN_wind_profile                         6641
GB_GBN_wind_offshore_capacity               6600
GB_GBN_wind_offshore_generation_actual        41
GB_GBN_wind_offshore_profile                6641
GB_GBN_wind_onshore_capacity                6600
GB_GBN_wind_onshore_generation_actual         41
GB_GBN_wind_onshore_profile                 6641
dtype: int64

In [62]:
#calculating the percentage of missing values in each column
(df.isnull().sum()/len(df))*100

utc_timestamp                                0.000000
GB_GBN_load_actual_entsoe_transparency       0.013889
GB_GBN_load_forecast_entsoe_transparency     0.240075
GB_GBN_price_day_ahead                       0.220234
GB_GBN_solar_capacity                       13.094978
GB_GBN_solar_generation_actual               0.109125
GB_GBN_solar_profile                        13.192199
GB_GBN_wind_capacity                        13.094978
GB_GBN_wind_generation_actual                0.081348
GB_GBN_wind_profile                         13.176326
GB_GBN_wind_offshore_capacity               13.094978
GB_GBN_wind_offshore_generation_actual       0.081348
GB_GBN_wind_offshore_profile                13.176326
GB_GBN_wind_onshore_capacity                13.094978
GB_GBN_wind_onshore_generation_actual        0.081348
GB_GBN_wind_onshore_profile                 13.176326
dtype: float64

In [63]:
missing_percentage=(df.isnull().sum()/len(df))*100
missing_percentage.sort_values(ascending=False)

GB_GBN_solar_profile                        13.192199
GB_GBN_wind_onshore_profile                 13.176326
GB_GBN_wind_offshore_profile                13.176326
GB_GBN_wind_profile                         13.176326
GB_GBN_wind_offshore_capacity               13.094978
GB_GBN_solar_capacity                       13.094978
GB_GBN_wind_capacity                        13.094978
GB_GBN_wind_onshore_capacity                13.094978
GB_GBN_load_forecast_entsoe_transparency     0.240075
GB_GBN_price_day_ahead                       0.220234
GB_GBN_solar_generation_actual               0.109125
GB_GBN_wind_generation_actual                0.081348
GB_GBN_wind_offshore_generation_actual       0.081348
GB_GBN_wind_onshore_generation_actual        0.081348
GB_GBN_load_actual_entsoe_transparency       0.013889
utc_timestamp                                0.000000
dtype: float64

In [64]:
#selecting the capacity columns
capacity_columns=["GB_GBN_solar_capacity","GB_GBN_wind_capacity","GB_GBN_wind_offshore_capacity","GB_GBN_wind_onshore_capacity"]

df[capacity_columns].isnull().all(axis=1).sum()

np.int64(6600)

In [65]:
missing_capacity_rows=df[capacity_columns].isnull().all(axis=1)
print(df.loc[missing_capacity_rows,"utc_timestamp"].head(10))

43801   2019-12-31 00:00:00+00:00
43802   2019-12-31 01:00:00+00:00
43803   2019-12-31 02:00:00+00:00
43804   2019-12-31 03:00:00+00:00
43805   2019-12-31 04:00:00+00:00
43806   2019-12-31 05:00:00+00:00
43807   2019-12-31 06:00:00+00:00
43808   2019-12-31 07:00:00+00:00
43809   2019-12-31 08:00:00+00:00
43810   2019-12-31 09:00:00+00:00
Name: utc_timestamp, dtype: datetime64[us, UTC]


In [66]:
print(df.loc[missing_capacity_rows,"utc_timestamp"].agg(["min","max"]))

min   2019-12-31 00:00:00+00:00
max   2020-09-30 23:00:00+00:00
Name: utc_timestamp, dtype: datetime64[us, UTC]


In [67]:
print(missing_capacity_rows.astype(int).diff().value_counts())

0.0    50399
1.0        1
Name: count, dtype: int64


In [68]:
print("First affected row:", missing_capacity_rows[missing_capacity_rows].index.min())
print("Last affected row:", missing_capacity_rows[missing_capacity_rows].index.max())
print("Total affected rows:", missing_capacity_rows.sum())

First affected row: 43801
Last affected row: 50400
Total affected rows: 6600


In [69]:
print(df["GB_GBN_solar_profile"].isnull().sum())

6649


In [70]:
solar_profile_missing = df["GB_GBN_solar_profile"].isnull()

print(solar_profile_missing.sum())
print(solar_profile_missing[solar_profile_missing].index.min())
print(solar_profile_missing[solar_profile_missing].index.max())

6649
0
50400


In [71]:
print(solar_profile_missing.astype(int).diff().value_counts())

GB_GBN_solar_profile
 0.0    50382
-1.0        9
 1.0        9
Name: count, dtype: int64


In [72]:
solar_changes = solar_profile_missing.astype(int).diff()
print(solar_changes[solar_changes != 0])

0        NaN
10      -1.0
3181     1.0
3188    -1.0
19562    1.0
19565   -1.0
23141    1.0
23143   -1.0
23237    1.0
23243   -1.0
23259    1.0
23263   -1.0
23412    1.0
23416   -1.0
24886    1.0
24893   -1.0
25187    1.0
25193   -1.0
43801    1.0
Name: GB_GBN_solar_profile, dtype: float64


In [73]:
print(df["GB_GBN_wind_profile"].isnull().sum())


6641


In [74]:
wind_profile_missing = df["GB_GBN_wind_profile"].isnull()
print(wind_profile_missing.sum())
print(wind_profile_missing[wind_profile_missing].index.min())
print(wind_profile_missing[wind_profile_missing].index.max())

6641
0
50400


In [75]:
# Check where missing-value status changes
print(wind_profile_missing.astype(int).diff().value_counts())

GB_GBN_wind_profile
 0.0    50382
-1.0        9
 1.0        9
Name: count, dtype: int64


In [76]:
wind_changes = wind_profile_missing.astype(int).diff()
print(wind_changes[wind_changes != 0])

0        NaN
2       -1.0
3181     1.0
3188    -1.0
19562    1.0
19565   -1.0
23141    1.0
23143   -1.0
23237    1.0
23243   -1.0
23259    1.0
23263   -1.0
23412    1.0
23416   -1.0
24886    1.0
24893   -1.0
25187    1.0
25193   -1.0
43801    1.0
Name: GB_GBN_wind_profile, dtype: float64


In [77]:
offshore_missing = df["GB_GBN_wind_offshore_profile"].isnull()
print(offshore_missing.sum())

6641


In [78]:
offshore_missing = df["GB_GBN_wind_offshore_profile"].isnull()
print(offshore_missing.astype(int).diff().value_counts())

GB_GBN_wind_offshore_profile
 0.0    50382
-1.0        9
 1.0        9
Name: count, dtype: int64


In [79]:
onshore_missing = df["GB_GBN_wind_onshore_profile"].isnull()
print(onshore_missing.sum())

6641


In [80]:
print(onshore_missing.astype(int).diff().value_counts())

GB_GBN_wind_onshore_profile
 0.0    50382
-1.0        9
 1.0        9
Name: count, dtype: int64


In [81]:
profile_columns =["GB_GBN_solar_profile","GB_GBN_wind_profile","GB_GBN_wind_offshore_profile","GB_GBN_wind_onshore_profile"]
df[profile_columns].isnull().all(axis=1).sum()

np.int64(6641)

In [82]:
missing_profile_rows = df[profile_columns].isnull().all(axis=1)
print((missing_capacity_rows & missing_profile_rows).sum())

6600


In [83]:
extra_profile_missing = missing_profile_rows & ~missing_capacity_rows
print(extra_profile_missing.sum())

41


In [84]:
df.loc[extra_profile_missing, capacity_columns].head(10)

,GB_GBN_solar_capacity,GB_GBN_wind_capacity,GB_GBN_wind_offshore_capacity,GB_GBN_wind_onshore_capacity
0,2664.0,11554.0,4040.0,7514.0
1,2669.0,11555.0,4040.0,7516.0
3181,4975.0,11926.0,4040.0,7886.0
3182,4975.0,11926.0,4040.0,7886.0
3183,4975.0,11926.0,4040.0,7886.0
3184,4975.0,11926.0,4040.0,7886.0
3185,4975.0,11926.0,4040.0,7886.0
3186,4975.0,11926.0,4040.0,7886.0
3187,4975.0,11926.0,4040.0,7886.0
19562,7811.0,15010.0,5094.0,9916.0


In [85]:
df.loc[extra_profile_missing, capacity_columns].isnull().sum()

GB_GBN_solar_capacity            0
GB_GBN_wind_capacity             0
GB_GBN_wind_offshore_capacity    0
GB_GBN_wind_onshore_capacity     0
dtype: int64

In [86]:
generation_columns = [
    col for col in df.columns
    if "generation" in col.lower()
]

print(generation_columns)

['GB_GBN_solar_generation_actual', 'GB_GBN_wind_generation_actual', 'GB_GBN_wind_offshore_generation_actual', 'GB_GBN_wind_onshore_generation_actual']


In [87]:
df[generation_columns].isnull().sum()

GB_GBN_solar_generation_actual            55
GB_GBN_wind_generation_actual             41
GB_GBN_wind_offshore_generation_actual    41
GB_GBN_wind_onshore_generation_actual     41
dtype: int64

In [88]:
(df[generation_columns].isnull().sum() / len(df)) * 100

GB_GBN_solar_generation_actual            0.109125
GB_GBN_wind_generation_actual             0.081348
GB_GBN_wind_offshore_generation_actual    0.081348
GB_GBN_wind_onshore_generation_actual     0.081348
dtype: float64

In [89]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().sum() / len(df)) * 100
})

missing_summary = missing_summary.sort_values(
    by="Missing_Count",
    ascending=False
)

print(missing_summary)

                                          Missing_Count  Missing_Percentage
GB_GBN_solar_profile                               6649           13.192199
GB_GBN_wind_onshore_profile                        6641           13.176326
GB_GBN_wind_offshore_profile                       6641           13.176326
GB_GBN_wind_profile                                6641           13.176326
GB_GBN_wind_offshore_capacity                      6600           13.094978
GB_GBN_solar_capacity                              6600           13.094978
GB_GBN_wind_capacity                               6600           13.094978
GB_GBN_wind_onshore_capacity                       6600           13.094978
GB_GBN_load_forecast_entsoe_transparency            121            0.240075
GB_GBN_price_day_ahead                              111            0.220234
GB_GBN_solar_generation_actual                       55            0.109125
GB_GBN_wind_generation_actual                        41            0.081348
GB_GBN_wind_

In [90]:
missing_forecast = df[
    "GB_GBN_load_forecast_entsoe_transparency"
].isnull()

print("Missing rows:", missing_forecast.sum())

print("First missing timestamp:")
print(df.loc[missing_forecast, "utc_timestamp"].min())

print("Last missing timestamp:")
print(df.loc[missing_forecast, "utc_timestamp"].max())

Missing rows: 121
First missing timestamp:
2014-12-31 23:00:00+00:00
Last missing timestamp:
2020-09-25 23:00:00+00:00


In [91]:
forecast_changes = missing_forecast.astype(int).diff()
print(forecast_changes.value_counts())

GB_GBN_load_forecast_entsoe_transparency
 0.0    50389
-1.0        6
 1.0        5
Name: count, dtype: int64


In [92]:
print(df.loc[missing_forecast, "utc_timestamp"].to_list())

[Timestamp('2014-12-31 23:00:00+0000', tz='UTC'), Timestamp('2019-05-18 00:00:00+0000', tz='UTC'), Timestamp('2019-05-18 01:00:00+0000', tz='UTC'), Timestamp('2019-05-18 02:00:00+0000', tz='UTC'), Timestamp('2019-05-18 03:00:00+0000', tz='UTC'), Timestamp('2019-05-18 04:00:00+0000', tz='UTC'), Timestamp('2019-05-18 05:00:00+0000', tz='UTC'), Timestamp('2019-05-18 06:00:00+0000', tz='UTC'), Timestamp('2019-05-18 07:00:00+0000', tz='UTC'), Timestamp('2019-05-18 08:00:00+0000', tz='UTC'), Timestamp('2019-05-18 09:00:00+0000', tz='UTC'), Timestamp('2019-05-18 10:00:00+0000', tz='UTC'), Timestamp('2019-05-18 11:00:00+0000', tz='UTC'), Timestamp('2019-05-18 12:00:00+0000', tz='UTC'), Timestamp('2019-05-18 13:00:00+0000', tz='UTC'), Timestamp('2019-05-18 14:00:00+0000', tz='UTC'), Timestamp('2019-05-18 15:00:00+0000', tz='UTC'), Timestamp('2019-05-18 16:00:00+0000', tz='UTC'), Timestamp('2019-05-18 17:00:00+0000', tz='UTC'), Timestamp('2019-05-18 18:00:00+0000', tz='UTC'), Timestamp('2019-05-

Missing values in the load forecast column will be retained because the missing observations include five complete 24-hour periods. Imputing these values without a reliable forecasting method could introduce inaccurate estimates into the dataset.

In [93]:
missing_price = df["GB_GBN_price_day_ahead"].isnull()

print("Missing values:", missing_price.sum())

print("First missing timestamp:")
print(df.loc[missing_price, "utc_timestamp"].min())

print("Last missing timestamp:")
print(df.loc[missing_price, "utc_timestamp"].max())

Missing values: 111
First missing timestamp:
2014-12-31 23:00:00+00:00
Last missing timestamp:
2020-09-30 23:00:00+00:00


In [94]:
price_changes = missing_price.astype(int).diff()
print(price_changes.value_counts())

GB_GBN_price_day_ahead
 0.0    50378
-1.0       11
 1.0       11
Name: count, dtype: int64


In [95]:
print(df.loc[missing_price, "utc_timestamp"].to_list())

[Timestamp('2014-12-31 23:00:00+0000', tz='UTC'), Timestamp('2015-01-01 00:00:00+0000', tz='UTC'), Timestamp('2015-01-01 01:00:00+0000', tz='UTC'), Timestamp('2015-01-01 02:00:00+0000', tz='UTC'), Timestamp('2015-01-01 03:00:00+0000', tz='UTC'), Timestamp('2015-01-01 04:00:00+0000', tz='UTC'), Timestamp('2015-01-01 05:00:00+0000', tz='UTC'), Timestamp('2015-01-01 06:00:00+0000', tz='UTC'), Timestamp('2015-01-01 07:00:00+0000', tz='UTC'), Timestamp('2015-01-01 08:00:00+0000', tz='UTC'), Timestamp('2015-01-01 09:00:00+0000', tz='UTC'), Timestamp('2015-01-01 10:00:00+0000', tz='UTC'), Timestamp('2015-01-01 11:00:00+0000', tz='UTC'), Timestamp('2015-01-01 12:00:00+0000', tz='UTC'), Timestamp('2015-01-01 13:00:00+0000', tz='UTC'), Timestamp('2015-01-01 14:00:00+0000', tz='UTC'), Timestamp('2015-01-01 15:00:00+0000', tz='UTC'), Timestamp('2015-01-01 16:00:00+0000', tz='UTC'), Timestamp('2015-01-01 17:00:00+0000', tz='UTC'), Timestamp('2015-01-01 18:00:00+0000', tz='UTC'), Timestamp('2015-01-

The missing values in the day-ahead electricity price column were retained because electricity prices fluctuate over time, and replacing missing observations with estimated values could misrepresent actual market prices. Since only approximately 0.22% of the values are missing, retaining them preserves the available data without introducing unsupported estimates

In [96]:
for col in generation_columns:
    print(f"\n{col}")
    print(df.loc[df[col].isnull(), "utc_timestamp"].to_list())


GB_GBN_solar_generation_actual
[Timestamp('2014-12-31 23:00:00+0000', tz='UTC'), Timestamp('2015-01-01 00:00:00+0000', tz='UTC'), Timestamp('2015-01-01 01:00:00+0000', tz='UTC'), Timestamp('2015-01-01 02:00:00+0000', tz='UTC'), Timestamp('2015-01-01 03:00:00+0000', tz='UTC'), Timestamp('2015-01-01 04:00:00+0000', tz='UTC'), Timestamp('2015-01-01 05:00:00+0000', tz='UTC'), Timestamp('2015-01-01 06:00:00+0000', tz='UTC'), Timestamp('2015-01-01 07:00:00+0000', tz='UTC'), Timestamp('2015-01-01 08:00:00+0000', tz='UTC'), Timestamp('2015-05-13 12:00:00+0000', tz='UTC'), Timestamp('2015-05-13 13:00:00+0000', tz='UTC'), Timestamp('2015-05-13 14:00:00+0000', tz='UTC'), Timestamp('2015-05-13 15:00:00+0000', tz='UTC'), Timestamp('2015-05-13 16:00:00+0000', tz='UTC'), Timestamp('2015-05-13 17:00:00+0000', tz='UTC'), Timestamp('2015-05-13 18:00:00+0000', tz='UTC'), Timestamp('2017-03-26 01:00:00+0000', tz='UTC'), Timestamp('2017-03-26 02:00:00+0000', tz='UTC'), Timestamp('2017-03-26 03:00:00+0000'

The missing values in the actual electricity generation columns will be retained rather than replaced with estimated or zero values. The missing observations occur at overlapping timestamps across multiple energy sources, suggesting that some periods lack recorded generation data. Replacing these values without reliable supporting information could misrepresent actual electricity generation.

In [97]:
missing_load = df["GB_GBN_load_actual_entsoe_transparency"].isnull()

print("Missing values:", missing_load.sum())

print(df.loc[missing_load, "utc_timestamp"].to_list())

Missing values: 7
[Timestamp('2014-12-31 23:00:00+0000', tz='UTC'), Timestamp('2015-03-29 01:00:00+0000', tz='UTC'), Timestamp('2015-03-29 02:00:00+0000', tz='UTC'), Timestamp('2015-03-29 03:00:00+0000', tz='UTC'), Timestamp('2017-03-26 01:00:00+0000', tz='UTC'), Timestamp('2017-03-26 02:00:00+0000', tz='UTC'), Timestamp('2017-03-26 03:00:00+0000', tz='UTC')]


In [98]:
# Create a copy of the original dataset
df_cleaned = df.copy()

# Display the dimensions of the treated dataset
print("Original dataset:", df.shape)
print("Cleaned dataset:", df_cleaned.shape)

Original dataset: (50401, 16)
Cleaned dataset: (50401, 16)


In [99]:
# Count missing values in the treated dataset
cleaned_missing = df_cleaned.isnull().sum()

print(cleaned_missing)

utc_timestamp                                  0
GB_GBN_load_actual_entsoe_transparency         7
GB_GBN_load_forecast_entsoe_transparency     121
GB_GBN_price_day_ahead                       111
GB_GBN_solar_capacity                       6600
GB_GBN_solar_generation_actual                55
GB_GBN_solar_profile                        6649
GB_GBN_wind_capacity                        6600
GB_GBN_wind_generation_actual                 41
GB_GBN_wind_profile                         6641
GB_GBN_wind_offshore_capacity               6600
GB_GBN_wind_offshore_generation_actual        41
GB_GBN_wind_offshore_profile                6641
GB_GBN_wind_onshore_capacity                6600
GB_GBN_wind_onshore_generation_actual         41
GB_GBN_wind_onshore_profile                 6641
dtype: int64


In [100]:
# Count missing values before treatment
before_missing = df.isnull().sum()

# Count missing values after treatment
after_missing = df_cleaned.isnull().sum()

# Create comparison table
comparison = pd.DataFrame({
    "Before_cleaning": before_missing,
    "After_cleaning": after_missing
})

print(comparison)

# Check whether missing-value counts changed
print("\nMissing-value counts unchanged:",
      before_missing.equals(after_missing))

                                          Before_cleaning  After_cleaning
utc_timestamp                                           0               0
GB_GBN_load_actual_entsoe_transparency                  7               7
GB_GBN_load_forecast_entsoe_transparency              121             121
GB_GBN_price_day_ahead                                111             111
GB_GBN_solar_capacity                                6600            6600
GB_GBN_solar_generation_actual                         55              55
GB_GBN_solar_profile                                 6649            6649
GB_GBN_wind_capacity                                 6600            6600
GB_GBN_wind_generation_actual                          41              41
GB_GBN_wind_profile                                  6641            6641
GB_GBN_wind_offshore_capacity                        6600            6600
GB_GBN_wind_offshore_generation_actual                 41              41
GB_GBN_wind_offshore_profile          

## Member 2: Data Types & Timestamp Formatting
This section applies **Member 2** formatting on top of the retained missing-value dataset (`df_cleaned`, `50,401` rows × `16` columns) before handing off to **Member 3** for duplicates, invalid values, and outliers. Missing values from Member 1's audit are intentionally preserved.

In [101]:
import pandas as pd

ts_col = "utc_timestamp"
ts_na_before = int(df_cleaned[ts_col].isna().sum())

df_cleaned[ts_col] = pd.to_datetime(df_cleaned[ts_col], utc=True, errors="coerce")
ts_na_after = int(df_cleaned[ts_col].isna().sum())

print(f"Timestamp NaNs before: {ts_na_before} -> after: {ts_na_after}")
print(f"Timestamp dtype: {df_cleaned[ts_col].dtype} | Timezone: {df_cleaned[ts_col].dt.tz}")
print(f"Duplicate timestamps: {int(df_cleaned[ts_col].duplicated().sum())}")
print(f"Monotonic increasing: {df_cleaned[ts_col].is_monotonic_increasing}")

Timestamp NaNs before: 0 -> after: 0
Timestamp dtype: datetime64[us, UTC] | Timezone: UTC
Duplicate timestamps: 0
Monotonic increasing: True


In [102]:
num_cols = [col for col in df_cleaned.columns if col != ts_col]
num_na_before = df_cleaned[num_cols].isna().sum()

for col in num_cols:
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors="coerce")

num_na_after = df_cleaned[num_cols].isna().sum()

type_audit = pd.DataFrame({
    "dtype_after": df_cleaned[num_cols].dtypes,
    "na_before": num_na_before,
    "na_after": num_na_after,
    "new_nans_from_coercion": num_na_after - num_na_before,
})

print("Numeric column type & coercion audit:")
print(type_audit)

Numeric column type & coercion audit:
                                         dtype_after  na_before  na_after  \
GB_GBN_load_actual_entsoe_transparency       float64          7         7   
GB_GBN_load_forecast_entsoe_transparency     float64        121       121   
GB_GBN_price_day_ahead                       float64        111       111   
GB_GBN_solar_capacity                        float64       6600      6600   
GB_GBN_solar_generation_actual               float64         55        55   
GB_GBN_solar_profile                         float64       6649      6649   
GB_GBN_wind_capacity                         float64       6600      6600   
GB_GBN_wind_generation_actual                float64         41        41   
GB_GBN_wind_profile                          float64       6641      6641   
GB_GBN_wind_offshore_capacity                float64       6600      6600   
GB_GBN_wind_offshore_generation_actual       float64         41        41   
GB_GBN_wind_offshore_profile          

In [103]:
assert df_cleaned.shape == (50401, 16), f"Unexpected shape: {df_cleaned.shape}"
assert str(df_cleaned[ts_col].dt.tz) == "UTC", "utc_timestamp is not in UTC"
assert df_cleaned[ts_col].isna().sum() == 0, "utc_timestamp has missing/unparsed values"
assert (type_audit["new_nans_from_coercion"] == 0).all(), (
    "Unexpected NaNs introduced during numeric coercion; inspect type_audit"
)
assert all(
    pd.api.types.is_float_dtype(df_cleaned[col]) or pd.api.types.is_integer_dtype(df_cleaned[col])
    for col in num_cols
), "One or more measurement columns are not numeric"

print("All Member 2 assertions passed. Ready for Member 3.")

All Member 2 assertions passed. Ready for Member 3.


### Member 2 Formatting Summary
- **Dataset Retained:** `50,401` rows × `16` columns (`df_cleaned`) with existing missing values kept intact.
- **Timestamps:** `utc_timestamp` parsed to UTC with 0 missing values and programmatically checked for duplicates and ordering.
- **Numeric Columns:** Measurement columns converted to numeric types via `pd.to_numeric(..., errors="coerce")` and audited for `new_nans_from_coercion` before `assert` checks.

## Member 3 – Duplicates, invalid values and outliers
This section works on the dataset produced by Members 1 and 2 (`df_cleaned`, 50,401 rows × 16 columns). It checks for duplicate records, impossible values and unusual observations. The unusual values are **flagged, not deleted**, because some may be genuine electricity-market conditions.

In [104]:
df = df_cleaned.copy()
#Shortening column names for simplification.
P = "GB_GBN_price_day_ahead"
LOAD = "GB_GBN_load_actual_entsoe_transparency"
SOLAR = "GB_GBN_solar_generation_actual"
SOLAR_CAP = "GB_GBN_solar_capacity"

In [105]:
#Locating unrealistic values:

#Outputting the number of duplicate rows and timestamps
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate timestamps:", df["utc_timestamp"].duplicated().sum())

#Columns with negative values
num = df.select_dtypes("number")
negatives = (num<0).sum()
print("Columns with negative values:")
print(negatives[negatives > 0])

#Locating Solar generation higher than solar capacity
print("Solar generation > solar capacity:", (df[SOLAR] > df[SOLAR_CAP]).sum())

Duplicate rows: 0
Duplicate timestamps: 0
Columns with negative values:
GB_GBN_price_day_ahead    81
dtype: int64
Solar generation > solar capacity: 233


In [106]:
#Checking profile values
profiles = [c for c in df.columns if c.endswith("_profile")]
rows = []
for c in profiles:
    rows.append({
        "columns":c,
        "below_0": int((df[c] <0).sum()),
        "above_1": int((df[c] >1).sum()),
        "max": round(df[c].max(),3),
    })
pd.DataFrame(rows)

,columns,below_0,above_1,max
0,GB_GBN_solar_profile,0,233,1.367
1,GB_GBN_wind_profile,0,0,0.755
2,GB_GBN_wind_offshore_profile,0,0,0.881
3,GB_GBN_wind_onshore_profile,0,0,0.760


In [107]:
#Unusual day ahead prices
print(df[P].describe().round(2))
print("Negative prices:", (df[P] <0).sum())
print("Prices >= 200:", (df[P] >= 200).sum())
print("Prices >= 500:", (df[P] >= 500).sum())

q1,q3 = df[P].quantile([0.25,0.75])
iqr = q3 - q1
lo,hi = q1 - 1.5 * iqr,q3 +1.5* iqr
print("IQR bounds:",round(lo,1), "to", round(hi,1))
print("Rows outside IQR bounds:",((df[P] < lo) | (df[P] > hi)).sum())


count    50290.00
mean        43.43
std         18.83
min        -38.80
25%         33.88
50%         41.01
75%         50.40
max        999.00
Name: GB_GBN_price_day_ahead, dtype: float64
Negative prices: 81
Prices >= 200: 29
Prices >= 500: 10
IQR bounds: 9.1 to 75.2
Rows outside IQR bounds: 1804


In [108]:
#10 most expensive hours
df.nlargest(10,P)[["utc_timestamp",P]]

,utc_timestamp,GB_GBN_price_day_ahead
14971,2016-09-15 18:00:00+00:00,999.00
15067,2016-09-19 18:00:00+00:00,960.00
14972,2016-09-15 19:00:00+00:00,799.00
16242,2016-11-07 17:00:00+00:00,796.00
16266,2016-11-08 17:00:00+00:00,775.92
15066,2016-09-19 17:00:00+00:00,750.00
15068,2016-09-19 19:00:00+00:00,747.00
16410,2016-11-14 17:00:00+00:00,700.00
16290,2016-11-09 17:00:00+00:00,600.01
15065,2016-09-19 16:00:00+00:00,500.00


In [109]:
#10 Least expensive hours
df.nsmallest(10,P)[["utc_timestamp",P]]

,utc_timestamp,GB_GBN_price_day_ahead
47271,2020-05-23 14:00:00+00:00,-38.80
47261,2020-05-23 04:00:00+00:00,-36.60
47260,2020-05-23 03:00:00+00:00,-34.93
47270,2020-05-23 13:00:00+00:00,-34.00
47269,2020-05-23 12:00:00+00:00,-28.80
47262,2020-05-23 05:00:00+00:00,-28.60
47259,2020-05-23 02:00:00+00:00,-26.05
47288,2020-05-24 07:00:00+00:00,-25.43
47285,2020-05-24 04:00:00+00:00,-24.56
47287,2020-05-24 06:00:00+00:00,-23.92


In [110]:
#Unusually low electricity demand
print("Load < 10,000 MW:",(df[LOAD] < 10000).sum())
print("Load < 15,000 MW:", (df[LOAD] < 15000).sum())
print("Mean load:",round(df[LOAD].mean()))
df.nsmallest(10,LOAD)[["utc_timestamp",LOAD]]

Load < 10,000 MW: 27
Load < 15,000 MW: 84
Mean load: 34921


,utc_timestamp,GB_GBN_load_actual_entsoe_transparency
38424,2019-05-20 23:00:00+00:00,511.0
38425,2019-05-21 00:00:00+00:00,539.0
38426,2019-05-21 01:00:00+00:00,575.0
39981,2019-07-24 20:00:00+00:00,757.0
39982,2019-07-24 21:00:00+00:00,852.0
38997,2019-06-13 20:00:00+00:00,874.0
39980,2019-07-24 19:00:00+00:00,926.0
38996,2019-06-13 19:00:00+00:00,1029.0
40005,2019-07-25 20:00:00+00:00,1282.0
40004,2019-07-25 19:00:00+00:00,1476.0


In [111]:
#Flag potential outliers
def flag_outliers(data):
    out = data.copy()
    q1, q3 = out[P].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr

    out["flag_price_negative"] = out[P] < 0
    out["flag_price_high"] = out[P] >= 500
    out["flag_price_iqr"] = (out[P] < lo) | (out[P] > hi)
    out["flag_solar_profile_gt1"] = out["GB_GBN_solar_profile"] > 1
    out["flag_solar_gt_capacity"] = out[SOLAR] > out[SOLAR_CAP]
    out["flag_load_low"] = out[LOAD] < 10000
    return out


def outlier_summary(data):
    cols = [c for c in data.columns if c.startswith("flag_")]
    s = data[cols].sum().rename("rows_flagged").to_frame()
    s["percent"] = (s["rows_flagged"] / len(data) * 100).round(2)
    return s

flagged = flag_outliers(df)
print("Shape before:", df.shape, "| after:", flagged.shape)
outlier_summary(flagged)

Shape before: (50401, 16) | after: (50401, 22)


,rows_flagged,percent
flag_price_negative,81,0.16
flag_price_high,10,0.02
flag_price_iqr,1804,3.58
flag_solar_profile_gt1,233,0.46
flag_solar_gt_capacity,233,0.46
flag_load_low,27,0.05


In [112]:
df_final = flag_outliers(df_cleaned)
print(df_cleaned.shape, "->", df_final.shape)   # (50401, 16) -> (50401, 22)
print("Rows flagged by at least one check:", df_final[[c for c in df_final if c.startswith("flag_")]].any(axis=1).sum())
outlier_summary(df_final)

(50401, 16) -> (50401, 22)
Rows flagged by at least one check: 2063


,rows_flagged,percent
flag_price_negative,81,0.16
flag_price_high,10,0.02
flag_price_iqr,1804,3.58
flag_solar_profile_gt1,233,0.46
flag_solar_gt_capacity,233,0.46
flag_load_low,27,0.05


## Cleaning decisions and justification 

| Issue | Rows | Decision | Justification |
|---|---|---|---|
| Duplicate rows / timestamps | 0 | No action | None found |
| Negative demand, generation, capacity | 0 | No action | None found |
| Negative day-ahead prices | 81 | Keep and flag | Negative prices occur when supply is higher than demand |
| Prices >= 500 | 10 | Keep and flag | Looks like a genuine market spike; to be verified |
| Prices outside IQR bounds | 1,804 | Keep and flag | Statistical outliers only; many are probably real market conditions |
| Solar profile > 1 / solar > capacity | 233 | Keep and flag | The profile is a calculated ratio; the capacity value may be outdated. Not capped |
| Load < 10,000 MW | 27 | Keep and flag | Possibly faulty readings; analysts should exclude flagged rows from share calculations |

### Member 3 summary

- **Duplicates:** no duplicates found (0 rows, 0 timestamps).
- **Invalid values:** no negative demand, generation or capacity; 233 solar profile values found that are above 1.
- **Outliers flagged:** 81 negative prices, 10 prices of 500 or more, 1,804 prices outside the IQR bounds, 233 solar rows above capacity or profile 1, and 27 low-load rows.
- **Decision:** all unusual rows were kept and flagged, and nothing was deleted or changed.
- **Output:** `df_final = flag_outliers(df_cleaned)` (50,401 rows × 22 columns) is passed to Member 5 for the final cleaned dataset.

## Member 5 ANALYTICAL QUESTIONS 4–6 AND CLEANING INTEGRATION

In [113]:
comparison = pd.DataFrame({
    "Measurement": [
        "Rows",
        "Columns",
        "Duplicate rows",
        "Missing values",
    ],

    "Original Dataset": [
        len(df),
        len(df.columns),
        df.duplicated().sum(),
        df.isna().sum().sum()
    ],

    "Final Cleaned Dataset": [
        len(df_cleaned),
        len(df_cleaned.columns),
        df_cleaned.duplicated().sum(),
        df_cleaned.isna().sum().sum()
    ]
})

print(comparison)

      Measurement  Original Dataset  Final Cleaned Dataset
0            Rows             50401                  50401
1         Columns                16                     16
2  Duplicate rows                 0                      0
3  Missing values             53389                  53389


In [114]:
print("Original shape:", df.shape)
print("Final shape:", df_cleaned.shape)

print(
    "Original columns preserved:",
    df.columns.equals(df_cleaned.columns)
)

print(
    "Missing values retained in the same positions:",
    df.isna().equals(df_cleaned.isna())
)

Original shape: (50401, 16)
Final shape: (50401, 16)
Original columns preserved: True
Missing values retained in the same positions: True


In [115]:
from pathlib import Path

out_dir = Path("..") / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)

In [116]:
df_final.to_csv(out_dir / "cleaned_data.csv", index = False)
print("Saved to:", out_dir / "cleaned_data.csv")

Saved to: ..\data\processed\cleaned_data.csv


In [117]:
df_final.to_pickle(out_dir / "cleaned_data.pkl")
print("Saved to:", out_dir / "cleaned_data.pkl")

Saved to: ..\data\processed\cleaned_data.pkl


In [118]:
# Reload the saved CSV to verify it
df_check = pd.read_csv(
    out_dir / "cleaned_data.csv",
    parse_dates=["utc_timestamp"]
)

print("Saved dataset shape:", df_check.shape)

print("\nMissing values:")
print(df_check.isna().sum())

print("\nData types:")
print(df_check.dtypes)

print("\nDuplicate rows:")
print(df_check.duplicated().sum())

Saved dataset shape: (50401, 22)

Missing values:
utc_timestamp                                  0
GB_GBN_load_actual_entsoe_transparency         7
GB_GBN_load_forecast_entsoe_transparency     121
GB_GBN_price_day_ahead                       111
GB_GBN_solar_capacity                       6600
GB_GBN_solar_generation_actual                55
GB_GBN_solar_profile                        6649
GB_GBN_wind_capacity                        6600
GB_GBN_wind_generation_actual                 41
GB_GBN_wind_profile                         6641
GB_GBN_wind_offshore_capacity               6600
GB_GBN_wind_offshore_generation_actual        41
GB_GBN_wind_offshore_profile                6641
GB_GBN_wind_onshore_capacity                6600
GB_GBN_wind_onshore_generation_actual         41
GB_GBN_wind_onshore_profile                 6641
flag_price_negative                            0
flag_price_high                                0
flag_price_iqr                                 0
flag_solar_profile_